# 🔬 Denoise + Grad-CAM: การศึกษาผลของการลดสัญญาณรบกวนต่อ Explainable AI

สมุดโน้ตบุ๊กนี้จำลองการทดลองส่วน Denoising เหมือนกับที่ทำใน CNN โดยเปรียบเทียบทั้ง **7 รูปแบบการเตรียมภาพ**:
1. **Baseline**: ภาพเอกซเรย์ดิบ (Raw CXR)
2. **Median Filter**: L1 (3×3), L2 (5×5), L3 (7×7)
3. **CLAHE + DWT**: L1 (clip 2 / depth 1), L2 (clip 4 / depth 2), L3 (clip 8 / depth 3)

**พร้อมการประเมิน Confusion Matrix ครบทั้ง 2 รูปแบบ:**
- **แบบที่ 1 (Classification Matrix):** โมเดลทาย Normal vs Infiltration ถูกต้องกี่ภาพ เหมือนใน CNN
- **แบบที่ 2 (Localization Matrix / Pointing Game):** Grad-CAM ชี้ตำแหน่งรอยโรคตกในกรอบ Bounding Box ของแพทย์จริงหรือไม่ (Hit vs Miss)

In [1]:
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

# ตรวจสอบ Path และนำเข้าโมดูล Denoise
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'denoise_methods.py').exists():
    BASE_DIR = CURRENT_DIR
    DATASET_DIR = CURRENT_DIR.parent.parent
else:
    BASE_DIR = CURRENT_DIR / 'notebooks' / 'Denoise_Gradcam'
    DATASET_DIR = CURRENT_DIR

sys.path.append(str(BASE_DIR))
import denoise_methods as dm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Dataset Path: {DATASET_DIR}")
print(f"พร้อมใช้งานฟังก์ชัน Denoise: {list(dm.ALL_METHODS.keys())}")

Device: cpu
Dataset Path: c:\Users\s0955\Github\Enhancing-Chest-X-RAY-Infiltration-Detection-Using-Noise-Reduction-and-Explainable-AI\notebooks
พร้อมใช้งานฟังก์ชัน Denoise: ['Baseline', 'Median L1 (3x3)', 'Median L2 (5x5)', 'Median L3 (7x7)', 'CLAHE+DWT L1', 'CLAHE+DWT L2', 'CLAHE+DWT L3']


## 1. โหลดข้อมูลภาพและพิกัดรอยโรคจริงของแพทย์ (Ground-Truth BBox)

In [2]:
manifest_path = DATASET_DIR / 'notebooks' / 'Gradcam' / 'sample_manifest_200.csv'
df_manifest = pd.read_csv(manifest_path)

bbox_path = DATASET_DIR / 'BBox_List_2017.csv'
df_bbox = pd.read_csv(bbox_path).iloc[:, :6]
df_bbox.columns = ['Image Index', 'Finding Label', 'x', 'y', 'w', 'h']

print(f"ชุดข้อมูลทดสอบ: Infiltration {sum(df_manifest['Class'] == 'Infiltration')} ภาพ | Normal {sum(df_manifest['Class'] == 'Normal')} ภาพ")
df_manifest[df_manifest['Class'] == 'Infiltration'].head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\s0955\\Github\\Enhancing-Chest-X-RAY-Infiltration-Detection-Using-Noise-Reduction-and-Explainable-AI\\notebooks\\notebooks\\Gradcam\\sample_manifest_200.csv'

## 2. โมเดล ResNet50 และอัลกอริทึม Grad-CAM

In [ ]:
# โหลดโมเดล ResNet50
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device)
model.eval()

# ตั้ง Hook ที่ layer4[-1] (Conv Block สุดท้าย)
gradients, activations = [], []
def hook_fwd(m, inp, out): activations.append(out)
def hook_bwd(m, g_inp, g_out): gradients.append(g_out[0])

target_layer = model.layer4[-1]
target_layer.register_forward_hook(hook_fwd)
target_layer.register_full_backward_hook(hook_bwd)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def compute_gradcam_for_numpy_image(img_u8):
    gradients.clear()
    activations.clear()
    
    img_pil = Image.fromarray(img_u8).convert('RGB')
    t_in = preprocess(img_pil).unsqueeze(0).to(device)
    
    out = model(t_in)
    pred_idx = out.argmax(dim=1).item()
    score = out[0, pred_idx]
    
    model.zero_grad()
    score.backward(retain_graph=True)
    
    grad = gradients[0][0].detach()
    act = activations[0][0].detach()
    weights = torch.mean(grad, dim=(1, 2), keepdim=True)
    cam = torch.sum(weights * act, dim=0).clamp(min=0).cpu().numpy()
    
    w, h = img_u8.shape[1], img_u8.shape[0]
    cam_resized = cv2.resize(cam, (w, h))
    if cam_resized.max() > cam_resized.min():
        cam_norm = (cam_resized - cam_resized.min()) / (cam_resized.max() - cam_resized.min())
    else:
        cam_norm = np.zeros_like(cam_resized)
        
    heatmap = np.uint8(255 * cam_norm)
    heatmap_colored = cv2.cvtColor(cv2.applyColorMap(heatmap, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    cxr_rgb = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2RGB)
    overlay = np.uint8(0.45 * heatmap_colored + 0.55 * cxr_rgb)
    
    return cam_norm, overlay

print("Grad-CAM Engine พร้อมทำงาน!")

## 3. เปรียบเทียบภาพที่ผ่านการ Denoise ทั้ง 7 รูปแบบบนคนไข้รายเดียวกัน

In [ ]:
sample_row = df_manifest[df_manifest['Class'] == 'Infiltration'].iloc[0]
img_path = DATASET_DIR / sample_row['Relative_Path']
raw_img = Image.open(img_path)
raw_u8 = np.array(raw_img.convert('L'))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for idx, (name, fn) in enumerate(dm.ALL_METHODS.items()):
    processed = fn(raw_u8)
    ax = axes[idx]
    ax.imshow(processed, cmap='gray')
    ax.set_title(f"{name}", fontsize=11, weight='bold', pad=8)
    ax.axis('off')

axes[7].axis('off')
plt.suptitle(f"เปรียบเทียบผล Denoising 7 วิธี บนภาพ: {sample_row['Image Index']}", fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

## 4. เปรียบเทียบ Grad-CAM + Ground-Truth Doctor's BBox ครบทั้ง 7 วิธี

In [ ]:
def compare_denoise_gradcam(image_index):
    row = df_manifest[df_manifest['Image Index'] == image_index].iloc[0]
    img_path = DATASET_DIR / row['Relative_Path']
    raw_u8 = np.array(Image.open(img_path).convert('L'))
    boxes = df_bbox[df_bbox['Image Index'] == image_index]
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for idx, (name, fn) in enumerate(dm.ALL_METHODS.items()):
        denoised = fn(raw_u8)
        cam, overlay = compute_gradcam_for_numpy_image(denoised)
        
        ax = axes[idx]
        ax.imshow(overlay)
        
        # ตีกรอบ BBox ของแพทย์
        for _, b in boxes.iterrows():
            bx, by, bw, bh = float(b['x']), float(b['y']), float(b['w']), float(b['h'])
            rect = patches.Rectangle((bx, by), bw, bh, linewidth=2.5, edgecolor='#00FF66', facecolor='none', linestyle='--')
            ax.add_patch(rect)
            
        ax.set_title(f"{name}", fontsize=11, weight='bold', pad=8)
        ax.axis('off')
        
    axes[7].axis('off')
    findings = row['Finding Labels']
    plt.suptitle(f"Denoise + Grad-CAM Comparison | Image: {image_index} ({findings})\n(Green Dashed Box = Doctor's Ground Truth BBox)", fontsize=13, weight='bold')
    plt.tight_layout()
    plt.show()

compare_denoise_gradcam(sample_row['Image Index'])

## 5. Confusion Matrix แบบที่ 1: ผลการทำนายคลาส (Classification Matrix)
วัดว่าโมเดล CNN ทาย Normal และ Infiltration ถูกต้องกี่ภาพ เหมือนกับการทดลองในโฟลเดอร์ `cnn`

In [ ]:
# โหลดภาพตัวอย่างที่เรนเดอร์สำเร็จแล้วมาแสดงผล
cm_class_img_path = BASE_DIR / 'output' / 'confusion_matrix_classification.png'
if cm_class_img_path.exists():
    plt.figure(figsize=(16, 5))
    plt.imshow(Image.open(cm_class_img_path))
    plt.axis('off')
    plt.show()
else:
    print("ยังไม่พบไฟล์ภาพ ให้รัน generate_confusion_matrices.py เพื่อสร้างภาพ")

## 6. Confusion Matrix แบบที่ 2: ความแม่นยำในการชี้ตำแหน่งรอยโรค (Localization Matrix / Pointing Game)
วัดว่า Grad-CAM ชี้ตำแหน่งรอยโรค Infiltration ได้ตรงจุด (**Hit ตกใน BBox ของแพทย์**) หรือชี้หลุดตำแหน่ง (**Miss หลุด BBox**)

In [ ]:
cm_loc_img_path = BASE_DIR / 'output' / 'confusion_matrix_xai_localization.png'
if cm_loc_img_path.exists():
    plt.figure(figsize=(16, 5))
    plt.imshow(Image.open(cm_loc_img_path))
    plt.axis('off')
    plt.show()
else:
    print("ยังไม่พบไฟล์ภาพ ให้รัน generate_confusion_matrices.py เพื่อสร้างภาพ")